# Mini-batch gradient descent

Diabetes data. Each update uses a small random group of rows.

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
print("OLS test R2", LinearRegression().fit(X_train, y_train).score(X_test, y_test))

## Our own mini-batch gradient descent

In [ ]:
class MBGDRegressor:
    def __init__(self, batch_size, learning_rate=0.01, epochs=100, seed=0):
        self.bs, self.lr, self.epochs = batch_size, learning_rate, epochs
        self.rng = np.random.default_rng(seed)

    def fit(self, X, y):
        n = X.shape[0]
        self.intercept_, self.coef_ = 0.0, np.ones(X.shape[1])
        for _ in range(self.epochs):
            order = self.rng.permutation(n)                # shuffle once per epoch
            for start in range(0, n, self.bs):
                j = order[start:start + self.bs]           # the next mini-batch of rows
                err = y[j] - (X[j] @ self.coef_ + self.intercept_)
                self.intercept_ -= self.lr * (-2 * err.mean())
                self.coef_ -= self.lr * (-2 * X[j].T @ err / len(j))
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

for bs in [1, 8, 32, 353]:
    m = MBGDRegressor(batch_size=bs, learning_rate=0.1, epochs=100).fit(X_train, y_train)
    print(f"batch size {bs:3d}: test R2 {r2_score(y_test, m.predict(X_test)):.3f}")

## scikit-learn: SGDRegressor has no batch_size, but partial_fit takes one mini-batch at a time

In [ ]:
warnings.simplefilter("ignore")
rng = np.random.default_rng(0)
sgd = SGDRegressor(learning_rate="constant", eta0=0.2, random_state=0)
n, batch_size = len(X_train), 35
for epoch in range(100):
    order = rng.permutation(n)
    for start in range(0, n, batch_size):
        j = order[start:start + batch_size]
        sgd.partial_fit(X_train[j], y_train[j])     # one pass over this mini-batch
print("test R2", round(sgd.score(X_test, y_test), 3))